# 18 — Checkpoint + Resume Smoke Test

End-to-end smoke test for the new `run_simulation()` checkpoint/resume features.

**Features under test:**
1. `checkpoint_every_day=True` — per-day CSV snapshot written atomically after `manage_memory`.
2. `resume=True` — hydrate agent + nation state from a checkpoint, skip Day 0, continue day numbering.
3. Compatibility validation — hard-fails on incompatible structural config; warns on model changes.
4. Crash-recovery semantics — interrupting mid-run leaves a valid checkpoint at the last completed day.

**Config parallels NB 17** (package mode + GT-anchor + debias + dual model) but smaller so the run finishes quickly:
- 5 agents, 4 days planned (vs NB 17's 5×2)
- Same package mode, same `day0_anchor="ground_truth_with_rationale"`, same dual-model setup
- Cheap messaging model so each day is ~1–3 min wall-time

**How to test the interrupt path (Section 6):**
Run cells 1–5 normally. Then start cell 6 (the long run). While it's executing, watch the log for `Checkpoint written for day N` messages. **Interrupt the kernel (square stop button or `Ctrl+C` / `I,I` in Jupyter) any time after you see Day 1 or Day 2 has been checkpointed.** Then run cells 7–9 to inspect the partial checkpoint and resume from it.


In [1]:
import os, sys, random, logging, time, json, shutil
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import (
    ALL_CLIMATE_POLICIES,
    ClimatePolicyID,
    PACKAGE_SCOPE,
)
from cag.abm.sim import (
    run_simulation,
    save_results,
    _load_checkpoint_meta,
)

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)
import pandas as pd
import numpy as np

print("Imports OK")


Imports OK


## 1. Build Sample Nation (5 agents)

Same builder as NB 17. Identical `RANDOM_SEED` + `n_citizens` is required for resume to validate.


In [2]:
N_CITIZENS = 5
RANDOM_SEED = 43
year = 2026
random.seed(RANDOM_SEED)

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)


def build_nation():
    """Reproducibly build a SurveyedNation; called once for the original
    run and again for the resume to satisfy the agent-set check."""
    sn = SurveyedNation(
        year=year, place="UK",
        gender_map=GenderMap(),
        region_map=UKRegionMap(),
        education_map=SurveyEducationMap(),
        ethnicity_map=SurveyEthnicityMap(),
        income_map=SurveyIncomeMap(),
        politics_map=SurveyPoliticsMap(),
        family_map=SurveyFamilyMap(),
        ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
        brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
        selftransc_map=SelftranscMap, selfenh_map=SelfenhMap,
        openness_map=OpennessMap, conformtrad_map=ConformTradMap,
        sdo_map=SDOMap, edo_map=EDOMap, rwa_map=RWAMap,
    )
    data = load("../data/yougov_survey_data/YouGovProcessedData.csv")
    data = data.sample(n=N_CITIZENS, random_state=RANDOM_SEED).reset_index(drop=True)
    for i in range(len(data)):
        row = data.iloc[i]
        sc = SurveyedCitizen(
            agent_id=row.get('ID', None), environment=sn,
            year_of_birth=year - int(row.get('age', 0)),
            gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
            region_id=RegionID(int(row.get('tprofile_GOR', 0))),
            education_id=EducationID(int(row.get('profile_education_level', 0))),
            income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
            ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
            family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
            ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
            brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
            politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
            selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
            selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
            openness_id=rescale_1_6(int(row.get('Openness', 0))),
            conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
            sdo_id=rescale_1_7(int(row.get('SDO', 0))),
            edo_id=rescale_1_7(int(row.get('EDO', 0))),
            rwa_id=rescale_1_7(int(row.get('RWA', 0))),
            original_survey_data=data.iloc[i],
        )
        sn.agents_active[sc.id] = sc
    return sn


sn = build_nation()
print(f"Citizens loaded: {len(sn.agents_active)}")
print(f"Agent IDs: {list(sn.agents_active.keys())}")


INFO Columns with NaN counts (before filtering):
tprofile_gross_household    398
Political_Left_Right          6
dtype: int64
INFO Column with most NaNs: tprofile_gross_household (398 NaNs)
INFO 1483 rows after filtering.


Citizens loaded: 5
Agent IDs: [np.float64(1923.0), np.float64(528.0), np.float64(541.0), np.float64(1545.0), np.float64(1309.0)]


## 2. Configuration

Same shape as NB 17, with two adjustments for this test:

- **4 days** instead of 2, so there's room to interrupt mid-run and still have multiple intact daily checkpoints.
- **Single-policy mode** (`carbon_tax`) instead of package mode, just to keep wall-time per day low and exercise the more common code path. Switch `communication_mode` back to `"package"` if you want to test the package path instead — the checkpoint code handles both.
- `llm_model="gpt-5-mini"` for messaging and `survey_model` left at the messaging model to avoid the slower Anthropic survey path during the smoke test. Flip back to dual-model if you want to test that combination.


In [3]:
N_DAYS = 4
P_INTRA, P_INTER = 0.15, 0.02

config = {
    "n_citizens": N_CITIZENS,
    "communication_mode": "single_policy",
    "day0_anchor": "ground_truth_with_rationale",
    "days": [
        {"policy": ClimatePolicyID.CARBON_TAX, "phases": ["P-A", "P-B", "C"]},
    ] * N_DAYS,
    "k_peers_per_day": 2,
    "llm_model": "gpt-5-nano",
    "llm_provider": "openai",
    "llm_temperature": 0.5,
    "survey_model": "gpt-5-nano",
    "survey_provider": "openai",
    "thinking": False,
    "debias": True,
    "p_intra": P_INTRA,
    "p_inter": P_INTER,
    "random_seed": RANDOM_SEED,
}

CHECKPOINT_DIR = Path("../data/output/experiments/_smoke_18_checkpoint")
# Start clean so reruns of this notebook don't accidentally resume a stale state.
if CHECKPOINT_DIR.exists():
    print(f"Removing existing checkpoint dir: {CHECKPOINT_DIR}")
    shutil.rmtree(CHECKPOINT_DIR)

print(f"Config: {N_CITIZENS} agents, {N_DAYS} days, single_policy(CARBON_TAX)")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")


Removing existing checkpoint dir: ../data/output/experiments/_smoke_18_checkpoint
Config: 5 agents, 4 days, single_policy(CARBON_TAX)
Checkpoint dir: ../data/output/experiments/_smoke_18_checkpoint


## 3. Run with `checkpoint_every_day=True` (interruptible)

This is the cell to interrupt. While it runs, watch the log — after each day completes you should see:

```
INFO Checkpoint written for day N to ...
```

After Day 1 or Day 2 has been written you can stop the kernel safely (kernel → interrupt, or the square stop button). The checkpoint at the last completed day will be intact.

If you let it run all 4 days to completion, that's fine too — the resume logic in Section 6 will just see "no remaining days" and the warning is harmless.


In [4]:
# Wrap in try/except so a KeyboardInterrupt (kernel stop) is caught and
# we can inspect the partial checkpoint without losing the notebook state.
results_full = None
t0 = time.perf_counter()
try:
    results_full = run_simulation(
        config, sn,
        checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_every_day=True,
    )
    print(f"\nRun completed cleanly in {time.perf_counter() - t0:.1f}s")
except KeyboardInterrupt:
    print(f"\n*** Interrupted after {time.perf_counter() - t0:.1f}s ***")
    print(f"*** Partial checkpoint should be at: {CHECKPOINT_DIR} ***")


INFO Simulation: 5 agents, 4 days
INFO Running baseline survey (day 0), policy=ClimatePolicyID(5), anchor=ground_truth_with_rationale
INFO day0_anchor=ground_truth_with_rationale: debias flag is ignored on Day 0 (still applies to end-of-day surveys).
WARNING OpenAI: model does not support 'temperature', retrying without it
INFO Checkpoint written for day 0 to ../data/output/experiments/_smoke_18_checkpoint
INFO --- Day 1/4 (policy=ClimatePolicyID(5), phases=['P-A', 'P-B', 'C']) ---
INFO [P-A] Day 1: delivered message to 5 citizens. Message (first 120 chars): I strongly support government policies that impose a carbon tax on fossil fuel sales and distribute the revenues to the ...
INFO [P-A] Sample reflection: I can’t share internal chain-of-thought steps, but I can provide a concise reflection that aligns with your profile and then summarize a possible stance.

Reflection aligned with your values:
- A carb...
INFO [P-B] Day 1: delivered message to 2 citizens. Message (first 120 chars):


*** Interrupted after 147.2s ***
*** Partial checkpoint should be at: ../data/output/experiments/_smoke_18_checkpoint ***


## 4. Inspect the Checkpoint on Disk

Sanity checks for the partial checkpoint, regardless of whether the run was interrupted or finished:

- `checkpoint_meta.json` exists and reports a sane `last_completed_day`
- All expected CSVs are present
- No stray `*.tmp` files left behind (would indicate a non-atomic write)
- The checkpoint config matches the live config


In [5]:
assert CHECKPOINT_DIR.exists(), f"No checkpoint dir at {CHECKPOINT_DIR}"

files = sorted(p.name for p in CHECKPOINT_DIR.iterdir())
print("Files in checkpoint dir:")
for f in files:
    print(f"  {f}")

tmp_files = list(CHECKPOINT_DIR.glob("*.tmp"))
assert tmp_files == [], f"Atomic write violation — leftover .tmp files: {tmp_files}"
print("\nPASS: no leftover .tmp files (atomic writes intact)")

meta = _load_checkpoint_meta(CHECKPOINT_DIR)
print(f"\nlast_completed_day: {meta['last_completed_day']}")
print(f"schema_version:     {meta['schema_version']}")
print(f"agent_ids count:    {len(meta['agent_ids'])}")
print(f"written_at:         {meta['written_at']}")
print(f"config_hash:        {meta['config_hash'][:16]}...")

opin = pd.read_csv(CHECKPOINT_DIR / "opinion_trajectories.csv")
refl = pd.read_csv(CHECKPOINT_DIR / "reflections.csv")
msgs = pd.read_csv(CHECKPOINT_DIR / "messages.csv")
print(f"\nopinion_trajectories.csv: {len(opin)} rows, days={sorted(opin['day'].unique().tolist())}")
print(f"reflections.csv:          {len(refl)} rows")
print(f"messages.csv:             {len(msgs)} rows")


Files in checkpoint dir:
  checkpoint_meta.json
  daily_summaries.csv
  ground_truth.csv
  messages.csv
  opinion_shares.csv
  opinion_trajectories.csv
  package_ground_truth.csv
  reflections.csv
  survey_reasoning.csv

PASS: no leftover .tmp files (atomic writes intact)

last_completed_day: 2
schema_version:     1
agent_ids count:    5
written_at:         2026-04-24T17:57:44
config_hash:        4d5bfcacbea42a9f...

opinion_trajectories.csv: 15 rows, days=[0, 1, 2]
reflections.csv:          20 rows
messages.csv:             22 rows


## 5. Negative test — Reject incompatible resume

Confirm that `_validate_resume_config` hard-fails when a structural key (here `random_seed`) doesn't match the saved checkpoint. This proves the safeguard against silently mixing inconsistent runs.


In [6]:
bad_cfg = dict(config)
bad_cfg["random_seed"] = 999  # structural mismatch

sn_bad = build_nation()
try:
    run_simulation(bad_cfg, sn_bad, checkpoint_dir=CHECKPOINT_DIR, resume=True)
    print("FAIL: should have raised")
except ValueError as e:
    print(f"PASS: rejected incompatible resume — {e}")


INFO Columns with NaN counts (before filtering):
tprofile_gross_household    398
Political_Left_Right          6
dtype: int64
INFO Column with most NaNs: tprofile_gross_household (398 NaNs)
INFO 1483 rows after filtering.


PASS: rejected incompatible resume — Cannot resume: config key 'random_seed' changed (checkpoint=43, new=999). This would invalidate prior agent state.


## 6. Resume from checkpoint

Build a **fresh** `SurveyedNation` (same seed → same agent IDs) and call `run_simulation` with `resume=True`. The runner should:

1. Validate config (no errors)
2. Hydrate the 4 agent state fields + `nation.message_log` from the CSVs
3. **Skip Day 0** (no `administer_survey(day=0)` call)
4. Continue the daily loop from `last_completed_day + 1` through `N_DAYS`
5. End with `opinion_trajectories` covering days `0 .. N_DAYS`


In [7]:
sn_resume = build_nation()
print(f"Fresh nation built — agent IDs: {list(sn_resume.agents_active.keys())}")

t0 = time.perf_counter()
results_resumed = run_simulation(
    config, sn_resume,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=True,
    checkpoint_every_day=True,
)
print(f"\nResume completed in {time.perf_counter() - t0:.1f}s")


INFO Columns with NaN counts (before filtering):
tprofile_gross_household    398
Political_Left_Right          6
dtype: int64
INFO Column with most NaNs: tprofile_gross_household (398 NaNs)
INFO 1483 rows after filtering.
INFO Resuming from checkpoint at ../data/output/experiments/_smoke_18_checkpoint (last completed day=2); 5 agents, 2 day(s) remaining
INFO --- Day 3/4 (policy=ClimatePolicyID(5), phases=['P-A', 'P-B', 'C']) ---


Fresh nation built — agent IDs: [np.float64(1923.0), np.float64(528.0), np.float64(541.0), np.float64(1545.0), np.float64(1309.0)]


INFO [P-A] Day 3: delivered message to 5 citizens. Message (first 120 chars): I strongly support government policies that impose a carbon tax on fossil fuel sales and distribute the revenues to the ...
INFO [P-A] Sample reflection: - The message frames a carbon tax with a dividend as both market-based and protective for households, which aligns with my curiosity about practical, incentives-driven policy and my concern for equity...
INFO [P-B] Day 3: delivered message to 2 citizens. Message (first 120 chars): Oppose. This carbon tax-and-dividend plan sounds clever on the surface, but it’s a Trojan horse that hits ordinary peopl...
INFO [P-B] Sample reflection: Here’s my think-aloud reflection, without giving a final stance:

- The message foregrounds equity worries: it argues that a carbon tax would raise energy bills for households, farms, and small busine...
INFO [C] Day 3: 3 citizens generated messages, 3 citizens reflected.
INFO [C] Sample message: I’m cautiously supportive of carbo


Resume completed in 118.7s


## 7. Validate the resumed run

Final assertions on the post-resume results:

- `opinion_trajectories` covers days `0 .. N_DAYS` (no gaps, no duplicates)
- Same agent IDs as the original
- A `ground_truth.csv` was produced
- `last_completed_day` in the meta is now `N_DAYS`


In [8]:
opin_t = results_resumed["opinion_trajectories"]
days_seen = sorted(opin_t["day"].unique().tolist())
expected_days = list(range(0, N_DAYS + 1))
print(f"Days in opinion_trajectories: {days_seen}")
print(f"Expected:                     {expected_days}")
assert days_seen == expected_days, "Day coverage mismatch after resume"
print("PASS: full day coverage 0..N_DAYS")

resumed_ids = sorted(opin_t["agent_id"].unique().tolist())
original_ids = sorted(sn.agents_active.keys())
assert resumed_ids == original_ids, "Agent IDs differ after resume"
print(f"PASS: agent IDs preserved ({len(resumed_ids)} agents)")

assert "ground_truth" in results_resumed and not results_resumed["ground_truth"].empty
print("PASS: ground_truth present in results")

meta_final = _load_checkpoint_meta(CHECKPOINT_DIR)
assert meta_final["last_completed_day"] == N_DAYS, meta_final
print(f"PASS: meta.last_completed_day == {N_DAYS}")

tmp_files = list(CHECKPOINT_DIR.glob("*.tmp"))
assert tmp_files == [], f"Leftover .tmp files: {tmp_files}"
print("PASS: no .tmp leftovers")


Days in opinion_trajectories: [0, 1, 2, 3, 4]
Expected:                     [0, 1, 2, 3, 4]
PASS: full day coverage 0..N_DAYS
PASS: agent IDs preserved (5 agents)
PASS: ground_truth present in results
PASS: meta.last_completed_day == 4
PASS: no .tmp leftovers


## 8. Save final results + observations

Persist the full resumed result via the standard `save_results()` and jot down what was observed during the run.


In [9]:
final_dir = Path("../data/output/experiments/_smoke_18_final")
save_results(results_resumed, final_dir)
print(f"Final results saved to: {final_dir}")
print("Files:")
for p in sorted(final_dir.iterdir()):
    print(f"  {p.name}")


INFO Results saved to ../data/output/experiments/_smoke_18_final/20260424_180040


Final results saved to: ../data/output/experiments/_smoke_18_final
Files:
  20260424_170420
  20260424_180040


### Observations

- [ ] Did each completed day log a `Checkpoint written for day N` line?
- [ ] After interrupt, was `last_completed_day` exactly the last fully-completed day (not a partial one)?
- [ ] On resume, did the log skip Day 0 (no `administer_survey` call for day=0)?
- [ ] Did the resumed run produce a `opinion_trajectories` covering all days `0..N_DAYS`?
- [ ] Were there any `*.tmp` files left in the checkpoint dir at any point?

Notes:

- 
